v62 
- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer

v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [1]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.builder",
    "core.contracts",    
    "core.engine",
    "core.environment",
    "core.features",
    "core.logic",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np
import multiprocessing
import gc

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
# from core.analyzer import create_walk_forward_analyzer
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic, SelectionLogic
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# # 7. Instantiate engine (customize DataFrames as needed)
# new_master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [ ]:
#### Change path to frozen data ####
DATA_PATH_OHLCV = r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet"
DATA_PATH_INDICES = (
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet"
)
print(f"=== Overwrite DATA_PATHS ===")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")

=== Overwrite DATA_PATHS ===
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet
DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet


In [5]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-18     25.00     26.57    24.82      26.56       0
       2026-03-19     27.96     28.03    25.19      25.54       0
       2026-03-20     26.07     28.61    25.91      27.43       0
       2026-03-23     25.60     26.59    24.73      26.10       0
       2026-03-24     26.86     27.20    25.79      26.56       0

[144848 rows x 5 columns]


In [6]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [7]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
# _indices = df_indices.index.get_level_values(0).unique().tolist()
# display(_indices)
# df_indices.info()

# # print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
# df_ohlcv.info()

In [ ]:
# print(f"features_df.info():\n{features_df.info()}\n")
# print(f"features_df.index.names:\n{features_df.index.names}\n")
# print(f"macro_df.info():\n{macro_df.info()}\n")
# print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [ ]:
# # Pre-flight Automated Audit Suite
# checks = [
#     SA.verify_math_integrity(),
#     SA.verify_ranking_integrity(),
#     SA.verify_vol_alignment_integrity(),
#     SA.verify_feature_engineering_integrity(),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

# print("=" * 85)
# # Separate verify_marco_engine output from intertwine with other outputs

# checks = [
#     SA.verify_macro_engine(
#         df_ohlcv=df_ohlcv,
#         df_indices=df_indices,
#         original_macro_df=macro_df,
#         settings=GLOBAL_SETTINGS,
#     ),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

In [8]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1585, Days: 16164
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [ ]:
print(f"df_close_wide.info():\n{df_close_wide.info()}")
print(f"df_atrp_wide.info():\n{df_atrp_wide.info()}")
print(f"df_trp_wide.info():\n{df_trp_wide.info()}")

In [9]:
# 6. Instantiate engine (customize DataFrames as needed)
new_master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

#### Run ParallelFeatureBuilder

In [ ]:
# # --- THE MARATHON CONFIG ---
CHECKPOINT_DIR = "_alpha_cache_v1_checkpoints"
FINAL_FILE = "AlphaCache_Master_2006_2026.parquet"
# # I recommend leaving 2 cores for your PC.
# # If you have an 8-core CPU, it will use 6 for the Bake.
WORKER_COUNT = max(1, multiprocessing.cpu_count() - 2)

ParallelFeatureBuilder.run_marathon(
    new_master_engine=new_master_engine,
    lookbacks=[21, 63, 189],
    start_date="2000-01-01",
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=50,
    num_workers=WORKER_COUNT,  # <--- TAME THE BEAST
)

# 2. THE STITCH (Run this when the progress bar hits 100%)
final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

#### Build final_cache_df 

In [21]:
# 2. THE STITCH (Run this when the progress bar hits 100%)
final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

🧵 Stitching 99 batches...


Merging:   0%|          | 0/99 [00:00<?, ?it/s]

✅ Final Master Cube Saved! Shape: (3596706, 39)


In [ ]:
list(final_cache_df.columns)

In [83]:
final_cache_df

21d_Price_Gain  21d_Sharpe  21d_Sharpe_ATRP  21d_Sharpe_TRP  21d_VIX_Filtered_Momentum_4  21d_Info_Ratio_Stdev_Alpha_63d  21d_Consistency_WinRate_5d  21d_Oversold_RSI  21d_Dip_Buyer_Drawdown_dd_21  21d_Low_Volatility_ATRP  21d_VIX_Filtered_Momentum_10  63d_Price_Gain  63d_Sharpe  63d_Sharpe_ATRP  63d_Sharpe_TRP  63d_VIX_Filtered_Momentum_15  63d_Info_Ratio_Stdev_Alpha_63d  63d_Consistency_WinRate_5d  63d_Oversold_RSI  63d_Dip_Buyer_Drawdown_dd_21  63d_Low_Volatility_ATRP  63d_VIX_Filtered_Momentum_21  189d_Price_Gain  189d_Sharpe  189d_Sharpe_ATRP  189d_Sharpe_TRP  189d_VIX_Filtered_Momentum_26  189d_Info_Ratio_Stdev_Alpha_63d  189d_Consistency_WinRate_5d  189d_Oversold_RSI  189d_Dip_Buyer_Drawdown_dd_21  189d_Low_Volatility_ATRP  189d_VIX_Filtered_Momentum_32  21d_Momentum_21d  21d_VIX_Filtered_Momentum  63d_Momentum_21d  63d_VIX_Filtered_Momentum  189d_Momentum_21d  189d_VIX_Filtered_Momentum
Date       Ticker                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
2006-09-01 A               1.4177      0.5299           1.0632          0.9987                       1.4509                         -0.8438                      0.6945           -0.0066                        0.1019                  -0.6468                        1.4509         -0.9184     -0.9494          -1.0099         -0.9895                        1.4509                         -0.8438                      0.6945           -0.0066                        0.1019                  -0.4265                        1.4509          -0.1765      -1.1683           -1.2464          -1.2237                         1.4509                          -0.8438                       0.6945            -0.0066                         0.1019                    0.0811                         1.4509               NaN                        NaN               NaN                        NaN                NaN                         NaN
           AA             -0.6993     -1.0715          -0.9158         -0.9546                      -0.7151                         -1.2877                     -1.2215            0.9882                       -0.0463                  -0.1240                       -0.7151         -0.8086     -1.1452          -1.0056         -1.0444                       -0.7151                         -1.2877                     -1.2215            0.9882                       -0.0463                  -0.3523                       -0.7151           0.0394      -0.2698           -0.3131          -0.3036                        -0.7151                          -1.2877                      -1.2215             0.9882                        -0.0463                   -0.3815                        -0.7151               NaN                        NaN               NaN                        NaN                NaN                         NaN
           AAL            -2.1313     -1.2538          -1.3380         -1.3623                      -2.0269                         -0.3036                     -0.2635            1.2815                        1.6348                  -2.8088                       -2.0269         -0.7389     -0.5951          -0.

In [82]:
final_cache_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3596706 entries, (Timestamp('2006-09-01 00:00:00'), 'A') to (Timestamp('2026-03-20 00:00:00'), 'ZTS')
Data columns (total 39 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   21d_Price_Gain                   float64
 1   21d_Sharpe                       float64
 2   21d_Sharpe_ATRP                  float64
 3   21d_Sharpe_TRP                   float64
 4   21d_VIX_Filtered_Momentum_4      float64
 5   21d_Info_Ratio_Stdev_Alpha_63d   float64
 6   21d_Consistency_WinRate_5d       float64
 7   21d_Oversold_RSI                 float64
 8   21d_Dip_Buyer_Drawdown_dd_21     float64
 9   21d_Low_Volatility_ATRP          float64
 10  21d_VIX_Filtered_Momentum_10     float64
 11  63d_Price_Gain                   float64
 12  63d_Sharpe                       float64
 13  63d_Sharpe_ATRP                  float64
 14  63d_Sharpe_TRP                   float64
 15  63d_VIX_Filtered_Momentum_15

In [ ]:
final_cache_df.describe()

#### Save final_cache_df 

In [23]:
# Get first and last dates from the index (level 0 is the date index)
first_date = final_cache_df.index.get_level_values(0).min().strftime("%Y%m%d")
last_date = final_cache_df.index.get_level_values(0).max().strftime("%Y%m%d")

# Construct filename with date range
fn_final_cache_df = OUTPUT_DIR / f"final_cache_df_{first_date}_{last_date}.parquet"
print(f"final_cache_df file name: {fn_final_cache_df}")

final_cache_df.to_parquet(fn_final_cache_df, engine="pyarrow", compression="zstd")
print(f"Saved final_cache_df to:\n{fn_final_cache_df}")

final_cache_df file name: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\final_cache_df_20060901_20260320.parquet
Saved final_cache_df to:
C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\final_cache_df_20060901_20260320.parquet


#### Verify saved final_cache_df 

In [24]:
# Load fn_final_cache_df to a different variable name
final_cache_df_reloaded = pd.read_parquet(fn_final_cache_df, engine="pyarrow")
print(f"\nLoaded: {fn_final_cache_df}\nTo: final_cache_df_reloaded\n")
print(f"final_cache_df_reloaded.head():\n{final_cache_df_reloaded.head()}")
print(f"final_cache_df_reloaded.tail():\n{final_cache_df_reloaded.tail()}")

# Verify data integrity
print(
    f"\nShape check - Original: {final_cache_df.shape}, Reloaded: {final_cache_df_reloaded.shape}"
)
print(f"Index match: {final_cache_df.index.equals(final_cache_df_reloaded.index)}")
print(
    f"Columns match: {final_cache_df.columns.equals(final_cache_df_reloaded.columns)}"
)

# Release from memory
del final_cache_df_reloaded

gc.collect()
print("\nReleased final_cache_df_reloaded from memory")


Loaded: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\final_cache_df_20060901_20260320.parquet
To: final_cache_df_reloaded

final_cache_df_reloaded.head():
                   21d_Price_Gain  21d_Sharpe  21d_Sharpe_ATRP  21d_Sharpe_TRP  21d_VIX_Filtered_Momentum_4  21d_Info_Ratio_Stdev_Alpha,_63d  21d_Consistency_WinRate_5d  21d_Oversold_RSI  21d_Dip_Buyer_Drawdown_dd_21  21d_Low_Volatility_ATRP  21d_VIX_Filtered_Momentum_10  63d_Price_Gain  63d_Sharpe  63d_Sharpe_ATRP  63d_Sharpe_TRP  63d_VIX_Filtered_Momentum_15  63d_Info_Ratio_Stdev_Alpha,_63d  63d_Consistency_WinRate_5d  63d_Oversold_RSI  63d_Dip_Buyer_Drawdown_dd_21  63d_Low_Volatility_ATRP  63d_VIX_Filtered_Momentum_21  189d_Price_Gain  189d_Sharpe  189d_Sharpe_ATRP  189d_Sharpe_TRP  189d_VIX_Filtered_Momentum_26  189d_Info_Ratio_Stdev_Alpha,_63d  189d_Consistency_WinRate_5d  189d_Oversold_RSI  189d_Dip_Buyer_Drawdown_dd_21  189d_Low_Volatility_ATRP  189d_VIX_Filtered_Momentum_32  21d_Momentum_21d  21d_VIX_F

In [ ]:
#############################
#############################

#### Load final_cache_df

In [44]:
fn_batch_df = r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\_alpha_cache_v1_checkpoints\batch_20260226.parquet"
# Load fn_batch_df to a different variable name
batch_df = pd.read_parquet(fn_batch_df, engine="pyarrow")
print(f"\nLoaded: {fn_batch_df}\nTo: batch_df\n")
print(f"batch_df.head():\n{batch_df.head()}")
print(f"batch_df.tail():\n{batch_df.tail()}")


Loaded: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\_alpha_cache_v1_checkpoints\batch_20260226.parquet
To: batch_df

batch_df.head():
                   21d_Price_Gain  21d_Sharpe  21d_Sharpe_ATRP  21d_Sharpe_TRP  21d_VIX_Filtered_Momentum_4  21d_Info_Ratio_Stdev_Alpha,_63d  21d_Consistency_WinRate_5d  21d_Oversold_RSI  21d_Dip_Buyer_Drawdown_dd_21  21d_Low_Volatility_ATRP  21d_VIX_Filtered_Momentum_10  63d_Price_Gain  63d_Sharpe  63d_Sharpe_ATRP  63d_Sharpe_TRP  63d_VIX_Filtered_Momentum_15  63d_Info_Ratio_Stdev_Alpha,_63d  63d_Consistency_WinRate_5d  63d_Oversold_RSI  63d_Dip_Buyer_Drawdown_dd_21  63d_Low_Volatility_ATRP  63d_VIX_Filtered_Momentum_21  189d_Price_Gain  189d_Sharpe  189d_Sharpe_ATRP  189d_Sharpe_TRP  189d_VIX_Filtered_Momentum_26  189d_Info_Ratio_Stdev_Alpha,_63d  189d_Consistency_WinRate_5d  189d_Oversold_RSI  189d_Dip_Buyer_Drawdown_dd_21  189d_Low_Volatility_ATRP  189d_VIX_Filtered_Momentum_32  21d_Momentum_21d  21d_VIX_Filtered_Momentum  63d_Momen

In [45]:
list(batch_df.columns)

['21d_Price_Gain',
 '21d_Sharpe',
 '21d_Sharpe_ATRP',
 '21d_Sharpe_TRP',
 '21d_VIX_Filtered_Momentum_4',
 '21d_Info_Ratio_Stdev_Alpha,_63d',
 '21d_Consistency_WinRate_5d',
 '21d_Oversold_RSI',
 '21d_Dip_Buyer_Drawdown_dd_21',
 '21d_Low_Volatility_ATRP',
 '21d_VIX_Filtered_Momentum_10',
 '63d_Price_Gain',
 '63d_Sharpe',
 '63d_Sharpe_ATRP',
 '63d_Sharpe_TRP',
 '63d_VIX_Filtered_Momentum_15',
 '63d_Info_Ratio_Stdev_Alpha,_63d',
 '63d_Consistency_WinRate_5d',
 '63d_Oversold_RSI',
 '63d_Dip_Buyer_Drawdown_dd_21',
 '63d_Low_Volatility_ATRP',
 '63d_VIX_Filtered_Momentum_21',
 '189d_Price_Gain',
 '189d_Sharpe',
 '189d_Sharpe_ATRP',
 '189d_Sharpe_TRP',
 '189d_VIX_Filtered_Momentum_26',
 '189d_Info_Ratio_Stdev_Alpha,_63d',
 '189d_Consistency_WinRate_5d',
 '189d_Oversold_RSI',
 '189d_Dip_Buyer_Drawdown_dd_21',
 '189d_Low_Volatility_ATRP',
 '189d_VIX_Filtered_Momentum_32',
 '21d_Momentum_21d',
 '21d_VIX_Filtered_Momentum',
 '63d_Momentum_21d',
 '63d_VIX_Filtered_Momentum',
 '189d_Momentum_21d',
 '

In [25]:
fn_final_cache_df = OUTPUT_DIR / "final_cache_df_20060901_20260320.parquet"
# Load fn_final_cache_df to a different variable name
final_cache_df = pd.read_parquet(fn_final_cache_df, engine="pyarrow")
print(f"\nLoaded: {fn_final_cache_df}\nTo: final_cache_df\n")
print(f"final_cache_df.head():\n{final_cache_df.head()}")
print(f"final_cache_df.tail():\n{final_cache_df.tail()}")


Loaded: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\final_cache_df_20060901_20260320.parquet
To: final_cache_df

final_cache_df.head():
                   21d_Price_Gain  21d_Sharpe  21d_Sharpe_ATRP  21d_Sharpe_TRP  21d_VIX_Filtered_Momentum_4  21d_Info_Ratio_Stdev_Alpha,_63d  21d_Consistency_WinRate_5d  21d_Oversold_RSI  21d_Dip_Buyer_Drawdown_dd_21  21d_Low_Volatility_ATRP  21d_VIX_Filtered_Momentum_10  63d_Price_Gain  63d_Sharpe  63d_Sharpe_ATRP  63d_Sharpe_TRP  63d_VIX_Filtered_Momentum_15  63d_Info_Ratio_Stdev_Alpha,_63d  63d_Consistency_WinRate_5d  63d_Oversold_RSI  63d_Dip_Buyer_Drawdown_dd_21  63d_Low_Volatility_ATRP  63d_VIX_Filtered_Momentum_21  189d_Price_Gain  189d_Sharpe  189d_Sharpe_ATRP  189d_Sharpe_TRP  189d_VIX_Filtered_Momentum_26  189d_Info_Ratio_Stdev_Alpha,_63d  189d_Consistency_WinRate_5d  189d_Oversold_RSI  189d_Dip_Buyer_Drawdown_dd_21  189d_Low_Volatility_ATRP  189d_VIX_Filtered_Momentum_32  21d_Momentum_21d  21d_VIX_Filtered_Momentum  

In [58]:
# Or if you want to modify in-place (more memory efficient)
final_cache_df.rename(columns=lambda x: x.replace(",", ""), inplace=True)
final_cache_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3596706 entries, (Timestamp('2006-09-01 00:00:00'), 'A') to (Timestamp('2026-03-20 00:00:00'), 'ZTS')
Data columns (total 39 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   21d_Price_Gain                   float64
 1   21d_Sharpe                       float64
 2   21d_Sharpe_ATRP                  float64
 3   21d_Sharpe_TRP                   float64
 4   21d_VIX_Filtered_Momentum_4      float64
 5   21d_Info_Ratio_Stdev_Alpha_63d   float64
 6   21d_Consistency_WinRate_5d       float64
 7   21d_Oversold_RSI                 float64
 8   21d_Dip_Buyer_Drawdown_dd_21     float64
 9   21d_Low_Volatility_ATRP          float64
 10  21d_VIX_Filtered_Momentum_10     float64
 11  63d_Price_Gain                   float64
 12  63d_Sharpe                       float64
 13  63d_Sharpe_ATRP                  float64
 14  63d_Sharpe_TRP                   float64
 15  63d_VIX_Filtered_Momentum_15

In [ ]:
######################
######################

In [50]:
# 6. Instantiate engine (customize DataFrames as needed)
old_master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [51]:
# 1. Define the Action (Inputs)
my_input = EngineInput(
    mode="Ranking",
    start_date=pd.Timestamp("2025-12-10"),  # Decision Date from your screenshot
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=10,
    debug=True,
)

# 2. Run the Headless Engine
metrics_df = run_headless_simulation(old_master_engine, my_input)

# 3. Verify Result
print("--- HEADLESS METRICS REPORT ---")
display(metrics_df)

# TEST: Gain in Holding Period (The Reward for RL)
reward = metrics_df.loc["Group Gain", "Holding"]
print(f"\nTarget Reward Signal: {reward:.4f}")

DEBUG: 939 stocks passed filters on 2025-12-10
----------------------------------------------------------------------
Timeline: [2025-11-10] -> Decision: 2025-12-10 -> Entry: 2025-12-11 -> End: 2025-12-18
Selected Tickers (10):
SHV, SGOV, EXAS, BIL, MINT, BOXX, USFR, PULS, RY, DG
----------------------------------------------------------------------
--- HEADLESS METRICS REPORT ---


,Full,Lookback,Holding
Metric,,,
Group Gain,0.1001,0.0882,0.0037
Benchmark Gain,-0.0073,0.0090,-0.0186
== Gain Delta,0.1074,0.0792,0.0223
Group Sharpe,9.1699,9.3898,28.2954
Benchmark Sharpe,-0.4945,0.9141,-7.6352
== Sharpe Delta,9.6644,8.4757,35.9305
Group Sharpe (ATRP),0.4525,0.5027,0.1178
Benchmark Sharpe (ATRP),-0.0200,0.0367,-0.3363
== Sharpe (ATRP) Delta,0.4725,0.4660,0.4540



Target Reward Signal: 0.0037


In [ ]:
######################
######################

In [66]:
class RLVRParityVerifier:
    @staticmethod
    def verify(
        cache_df: pd.DataFrame,
        df_close_wide: pd.DataFrame,
        engine_output: any,
        lookback: int,
        registry_metric: str,
        top_n: int = 10,
    ):
        # 1. Modular Name Mapping
        # Slugify returns a list, so we take the first element
        clean_metric = AlphaLogic.slugify_columns([registry_metric])[0]
        print(f"\nregistry_metric: {registry_metric}")
        print(f"clean_metric: {clean_metric}")

        cache_col = f"{lookback}d_{clean_metric}"
        decision_date = engine_output.decision_date
        print(f"cache_col: {cache_col}")
        print(f"decision_date: {decision_date}")

        # 2. Selection Parity
        day_slice = cache_df.xs(decision_date, level="Date")[cache_col]
        cache_tickers = (
            day_slice.sort_values(ascending=False).head(top_n).index.tolist()
        )
        print(f"day_slice cache_df: {day_slice}")
        print(f"cache_tickers: {cache_tickers}")

        engine_tickers = engine_output.tickers
        print(f"engine_tickers: {engine_tickers}")

        # 3. REWARD LOGIC: REPLICATING QuantUtils.compute_portfolio_stats
        # Entry (T+1) to End (T+1+H)
        entry_date = engine_output.buy_date
        end_date = engine_output.holding_end_date

        # Slice the window [Entry : End]
        price_window = df_close_wide.loc[entry_date:end_date, cache_tickers]
        print(f"price_window cache_tickers:\n{price_window}\n")

        # a) Normalize prices to the START of the holding period (Entry Date)
        # This is what norm_prices = prices.div(prices.bfill().iloc[0]) does
        norm_prices = price_window.div(price_window.iloc[0])

        # b) Equal weights at the start (1/N)
        weights = 1.0 / len(cache_tickers)

        # c) Equity Curve = Sum of (Relative Price * Weight)
        # This accounts for the "Price Drift" within the holding period
        equity_curve = (norm_prices * weights).sum(axis=1)

        # d) Calculate Total Period Log Return (Veritable Reward)
        final_equity = equity_curve.iloc[-1]
        vectorized_log_reward = np.log(final_equity)

        # 4. Comparison
        legacy_reward = engine_output.perf_metrics.get("holding_p_gain", 0.0)
        gain_diff = abs(vectorized_log_reward - legacy_reward)

        print(f"--- 100% PARITY REPORT: {decision_date.date()} ---")
        print(
            f"Ticker Match:         {'✅ PASS' if set(cache_tickers) == set(engine_tickers) else '❌ FAIL'}"
        )
        print(f"Log Reward Match:     {'✅ PASS' if gain_diff < 1e-9 else '❌ FAIL'}")
        print(f"  - Vectorized (Log): {vectorized_log_reward:.10f}")
        print(f"  - Legacy (Log):     {legacy_reward:.10f}")
        print(f"  - Delta:            {gain_diff:.2e}")


# # Run the test again
# RLVRParityVerifier.verify(
#     cache_df=final_cache_df,
#     df_close_wide=df_close_wide,
#     engine_output=engine_output,
#     lookback=21,
#     registry_metric="Sharpe (ATRP)",
#     top_n=10,
# )

In [80]:
# for key in [
#     "Sharpe (TRP)",
#     "Momentum (21d)",
#     "Consistency (WinRate 5d)",
#     "Low Volatility (-ATRP)",
#     "VIX Filtered Momentum",
# ]:
for key in METRIC_REGISTRY.keys():
    # 1. Update the input metric to TRP
    my_input.metric = key
    my_input.rank_start = 1
    my_input.rank_end = 1
    print(f"my_input.metric: {my_input.metric}")

    # 2. RE-RUN the engine to get a FRESH output for TRP
    engine_output = old_master_engine.run(my_input)

    # 3. NOW run the parity check with the matching output
    RLVRParityVerifier.verify(
        cache_df=final_cache_df,
        df_close_wide=df_close_wide,
        engine_output=engine_output,  # Use the new output
        lookback=21,
        registry_metric=my_input.metric,
        top_n=1,
    )
    print(f'{"="*10}\n')

my_input.metric: Price Gain
DEBUG: 939 stocks passed filters on 2025-12-10

registry_metric: Price Gain
clean_metric: Price_Gain
cache_col: 21d_Price_Gain
decision_date: 2025-12-10 00:00:00
day_slice cache_df: Ticker
A      -0.4260
AA      1.4132
AAL     1.2797
AAPL    0.1938
ABBV    0.1370
         ...  
ZBH     0.3526
ZBRA    0.2627
ZM      0.4578
ZS     -3.6543
ZTS    -0.3632
Name: 21d_Price_Gain, Length: 937, dtype: float64
cache_tickers: ['EXAS']
engine_tickers: ['EXAS']
price_window cache_tickers:
Ticker        EXAS
Date              
2025-12-11  101.36
2025-12-12  101.50
2025-12-15  101.74
2025-12-16  101.76
2025-12-17  101.61
2025-12-18  101.38

--- 100% PARITY REPORT: 2025-12-10 ---
Ticker Match:         ✅ PASS
Log Reward Match:     ✅ PASS
  - Vectorized (Log): 0.0001972970
  - Legacy (Log):     0.0001972970
  - Delta:            0.00e+00

my_input.metric: Sharpe
DEBUG: 939 stocks passed filters on 2025-12-10

registry_metric: Sharpe
clean_metric: Sharpe
cache_col: 21d_Sharpe


In [81]:
# for key in [
#     "Sharpe (TRP)",
#     "Momentum (21d)",
#     "Consistency (WinRate 5d)",
#     "Low Volatility (-ATRP)",
#     "VIX Filtered Momentum",
# ]:
for key in METRIC_REGISTRY.keys():
    # 1. Update the input metric to TRP
    my_input.metric = key
    my_input.rank_start = 1
    my_input.rank_end = 1
    print(f"my_input.metric: {my_input.metric}")

    # 2. RE-RUN the engine to get a FRESH output for TRP
    engine_output = old_master_engine.run(my_input)

    # 3. NOW run the parity check with the matching output
    RLVRParityVerifier.verify(
        cache_df=final_cache_df,
        df_close_wide=df_close_wide,
        engine_output=engine_output,  # Use the new output
        lookback=63,
        registry_metric=my_input.metric,
        top_n=1,
    )
    print(f'{"="*10}\n')

my_input.metric: Price Gain
DEBUG: 939 stocks passed filters on 2025-12-10

registry_metric: Price Gain
clean_metric: Price_Gain
cache_col: 63d_Price_Gain
decision_date: 2025-12-10 00:00:00
day_slice cache_df: Ticker
A       0.5854
AA      1.7628
AAL     0.7625
AAPL    0.9795
ABBV    0.0180
         ...  
ZBH    -0.8062
ZBRA   -1.0509
ZM      0.0693
ZS     -1.1184
ZTS    -1.5641
Name: 63d_Price_Gain, Length: 937, dtype: float64
cache_tickers: ['SNDK']
engine_tickers: ['EXAS']
price_window cache_tickers:
Ticker        SNDK
Date              
2025-12-11  241.61
2025-12-12  206.18
2025-12-15  201.87
2025-12-16  209.31
2025-12-17  206.83
2025-12-18  219.46

--- 100% PARITY REPORT: 2025-12-10 ---
Ticker Match:         ❌ FAIL
Log Reward Match:     ❌ FAIL
  - Vectorized (Log): -0.0961548724
  - Legacy (Log):     0.0001972970
  - Delta:            9.64e-02

my_input.metric: Sharpe
DEBUG: 939 stocks passed filters on 2025-12-10

registry_metric: Sharpe
clean_metric: Sharpe
cache_col: 63d_Sharpe

In [ ]:
# Run the test again
RLVRParityVerifier.verify(
    cache_df=final_cache_df,
    df_close_wide=df_close_wide,
    engine_output=engine_output,
    lookback=21,
    registry_metric="Sharpe (ATRP)",
    top_n=10,
)

In [ ]:
final_cache_df.columns

In [ ]:
# 1. Update the input metric to TRP
my_input.metric = "Sharpe (TRP)"
print(f"my_input.metric: {my_input.metric}\n")

# 2. RE-RUN the engine to get a FRESH output for TRP
engine_output_trp = old_master_engine.run(my_input)

# 3. NOW run the parity check with the matching output
RLVRParityVerifier.verify(
    cache_df=final_cache_df,
    df_close_wide=df_close_wide,
    engine_output=engine_output_trp,  # Use the new output
    lookback=21,
    registry_metric=my_input.metric,
    top_n=10,
)

In [ ]:
######################
######################

In [ ]:
# List of metric names
metric_names = list(METRIC_REGISTRY.keys())
print(f"metric_names: {metric_names}")

In [ ]:
decision_date = pd.Timestamp("2025-12-10")
# decision_date = pd.Timestamp("2025-12-07")
# start_date = pd.Timestamp("2025-11-25")
holding_period = 5
lookback_period = 21
metric = "Sharpe (ATRP)"
rank_start = 11
rank_end = 12
benchmark_ticker = GLOBAL_SETTINGS["benchmark_ticker"]
thresholds = GLOBAL_SETTINGS["thresholds"]

calendar = new_master_engine.trading_calendar

# Use searchsorted to find the insertion point (next date if exact not found)
decision_idx = calendar.searchsorted(decision_date)

# Safety check: ensure we haven't requested a date beyond the calendar
if decision_idx >= len(calendar):
    raise ValueError(
        f"Decision date {decision_date.date()} is beyond the last trading date "
        f"({calendar[-1].date()})"
    )

actual_trading_date = calendar[decision_idx]

# Only print and replace if the requested date is not in the index
if actual_trading_date != decision_date:
    print(
        f"=== Using decision_date: {actual_trading_date.date()} ===\n"
        f"=== Requested decision_date: {decision_date.date()} is not in index ==="
    )
    decision_date = actual_trading_date

# Compare with Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker=benchmark_ticker,
    rank_start=rank_start,
    rank_end=rank_end,
    # start_date is decision_date
    start_date=decision_date,
    holding_period=holding_period,
    lookback_period=lookback_period,
    metric=metric,
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# # AlphaCache_Final run parameters
# lookback_periods = [21, 63, 189]
# lookback_period = lookback_periods[0]
# holding_period = 5
# metric = "Sharpe (ATRP)"

In [ ]:
engine_out.perf_metrics
engine_out.tickers
# engine_out

In [ ]:
# 1. First, get the 'Survivor Pool' (The candidates) for this specific date
# This uses your verified liquidity/quality thresholds
candidates = new_master_engine._filter_universe(
    date_ts=decision_date,
    thresholds=thresholds,
    audit_container={},  # We pass an empty dict for headless runs
)
print(f"len(candidates): {len(candidates)}")

In [ ]:
# # decision_date = pd.Timestamp("2025-01-11")
# decision_idx = new_master_engine.trading_calendar.get_loc(decision_date)
# decision_idx

In [ ]:
# --- THE PRECISION PATTERN ---

# 1. Find where 'Today' (Decision Date) sits in the calendar
decision_idx = new_master_engine.trading_calendar.get_loc(decision_date)

# 2. Go back exactly 21 steps to find the Anchor (P0)
# This handles holidays (Christmas, New Year) automatically
start_idx = decision_idx - lookback_period
start_date = new_master_engine.trading_calendar[start_idx]

# print(f"Decision Date: {pd.Timestamp('2025-01-13').date()}")
print(f"Decision Date: {decision_date.date()}")
print(f"Precise P0 Anchor (21 days back): {start_date.date()}")

In [ ]:
print(f"decision_date: {decision_date.date()}")
print(f"start_date: {start_date.date()}")
print(f"len(candidates): {len(candidates)}")

In [ ]:
# 1. Initialize our storage for this date's ensemble
ensemble_parts = []

# 2. Outer Loop: Resolutions (Lookbacks)
# for lb in [21, 63, 189]:
for lb in [lookback_period]:
    # Generate the 'MarketObservation' for this specific resolution
    # This is where 'obs' is born
    # obs = new_master_engine._build_observation(date, candidates, start_date_for_lb)
    obs = new_master_engine._build_observation(
        decision_date=decision_date, candidates=candidates, start_date=start_date
    )

    # 3. Inner Loop: The Metric Registry (The "Brain" Loop)
    for metric_name, metric_func in METRIC_REGISTRY.items():

        # --- THIS IS THE CALL ---
        # metric_func is the reference from the dictionary
        # obs is the dataclass containing prices, returns, ATRP, etc.
        score_series = metric_func(obs)

        # 4. Standardize the Column Name for the Agent
        # Example: '21d_Sharpe_ATRP'
        col_name = f"{lb}d_{metric_name}"
        score_series.name = col_name

        ensemble_parts.append(score_series)

# 5. Combine into a Single Matrix [Tickers x 33 Metrics]
ensemble_df = pd.concat(ensemble_parts, axis=1)

# 6. Apply to ensemble_df columns (static method called from outside the class)
ensemble_df.columns = AlphaLogic.slugify_columns(ensemble_df.columns.tolist())

In [ ]:
print(f"ensemble_df:\n{ensemble_df}")

In [ ]:
# ensemble_df
type(obs)
obs.lookback_returns.A

In [ ]:
# --- THE FORENSIC PRINT PATTERN ---

for metric_name, metric_func in METRIC_REGISTRY.items():
    # Execute the vectorized strategy
    scores = metric_func(obs)

    # 1. Look at the aggregate (The 'Market Weather')
    avg_val = scores.mean()
    count = len(scores)

    # 2. Look at the leaders (The 'Top Alpha')
    top_3 = scores.sort_values(ascending=False).head(3)

    print(f"📊 Strategy: {metric_name} ({count} tickers)")
    print(f"   Market Average: {avg_val:+.4f}")
    print(
        f"   Top 3 Tickers:  {top_3.index.tolist()} -> {top_3.values} -> {top_3.values.mean()}"
    )
    print("-" * 30)

In [ ]:
# Output file name for AlphaCache
fn_AlphaCache = OUTPUT_DIR / "AlphaCache_Final.parquet"
print(f"AlphaCache file name: {fn_AlphaCache}")

In [ ]:
# Load AlphaCache_Final.parquet
fn_AlphaCache = OUTPUT_DIR / "AlphaCache_Final.parquet"
final_cache_df = pd.read_parquet(fn_AlphaCache, engine="pyarrow")
print(f"\nLoaded: {fn_AlphaCache}\nTo: final_cache_df\n")
print(f"final_cache_df.head():\n{final_cache_df.head()}")
print(f"final_cache_df.tail():\n{final_cache_df.tail()}")

In [ ]:
print(f"final_cache_df columns:")
for index, value in enumerate(final_cache_df.columns):
    print(f"{index}: {value}")

In [ ]:
print(f"lookback_periods: {lookback_periods}")
print(f"start_date: {start_date.date()}")

In [ ]:
# --- THE MARATHON CONFIG ---
CHECKPOINT_DIR = "alpha_cache_v1_checkpoints"
FINAL_FILE = "AlphaCache_Master_2000_2026.parquet"

# 1. THE BAKE (Parallel + Resilient)
ParallelFeatureBuilder.run_marathon(
    new_master_engine=new_master_engine,
    lookbacks=[21, 63, 189],
    start_date="2000-01-01",
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=10,  # Saves to disk every 50 days
)

# 2. THE STITCH (Run this when the progress bar hits 100%)
# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

In [ ]:
# Create AlphaCache_Final takes 1.5 hour run
# 1. BAKE THE FINAL CUBE (The 1.5 Hour Re-run)
final_cache_df = FeatureCubeBuilder.build(
    new_master_engine=new_master_engine,
    lookbacks=lookback_periods,
    start_date="2000-01-01",
)

final_cache_df.to_parquet(fn_AlphaCache, engine="pyarrow", compression="zstd")
print(f"Saved final_cache_df to:\n{fn_AlphaCache}")

In [ ]:
new_master_engine.compute_alpha_ensemble(
    decision_date=start_date,
    lookback_periods=lookback_periods,
)

In [ ]:
# 1. Choose your Discovery resolution (Standard: 5 days)
# holding_pd = 5

# 2. Command the Engine to calculate the 'Forward-Truth'
# This generates the reward_matrix [Dates x Tickers]
# Logic: ln(Price_t+6 / Price_t+1)
new_master_engine.precompute_reward_matrix(holding_period=holding_period)

print(f"✅ Reward Matrix Generated. Shape: {new_master_engine.reward_matrix.shape}")

# 3. NOW Initialize the Environment
# It will now find 'new_master_engine.reward_matrix' and work perfectly.
env = DiscoveryEnv(
    feature_cube=final_cache_df,
    reward_matrix=new_master_engine.reward_matrix,
    calendar=new_master_engine.trading_calendar,
    holding_period=holding_period,
)

print("🚀 DiscoveryEnv Ready. Proceed to One-Hot Audit.")

In [ ]:
engine_out.perf_metrics

In [ ]:
# # 1. BAKE THE FINAL CUBE (The 1.5 Hour Re-run)
# # Use start_date="2024-12-10" as we discussed
# final_cache_df = FeatureCubeBuilder.build(
#     new_master_engine, [21, 63, 189], pd.Timestamp("2024-12-10")
# )
# # final_cache_df.to_pickle("AlphaCache_Final_Slugified.pkl")
# final_cache_df.to_parquet(f_name, engine="pyarrow", compression="zstd")
# print(f"Saved final_cache_df to:\n{f_name}")

# 2. INITIALIZE THE NEW ENVIRONMENT
env = DiscoveryEnv(
    final_cache_df, new_master_engine.reward_matrix, new_master_engine.trading_calendar
)

# 3. RUN ONE-HOT AUDIT
# test_date = pd.Timestamp("2025-01-02")
target_metric = "21d_Sharpe_ATRP"  # Note: Slugified name!
metric_idx = final_cache_df.columns.get_loc(target_metric)

# Build One-Hot Action
action = np.zeros(final_cache_df.shape[1] + 2)
action[metric_idx] = 1.0
action[-2], action[-1] = -1.0, 1.0  # Offset 0, Width 10

env.reset(start_date=start_date)
_, env_reward, _, info = env.step(action)

# Compare with Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker="SPY",
    start_date=start_date,
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# SAVE EXCEL AUDIT
audit_df = engine_out.debug_data["selection_audit"].join(
    final_cache_df.xs(start_date, level="Date")[[target_metric]], how="inner"
)

fn_audit_df_csv = OUTPUT_DIR / "Final_Absolute_Zero_Audit.csv"
audit_df.to_csv(fn_audit_df_csv)
print(f"Save audit_df to: {fn_audit_df_csv}")

engine_reward = engine_out.perf_metrics["holding_p_gain"]
reward_diff = engine_reward - env_reward
print(f"✅ Audit Generated. Open 'Final_Absolute_Zero_Audit.csv'")
print(f"Env Reward: {env_reward:.10f}")
print(f"Engine Reward: {engine_reward:.10f}")
print(f"Reward Diff: {reward_diff:.10f}")

In [ ]:
# 1. Define the Discovery Goal (5-Day Holding)
holding_pd = 5

# 2. Precompute the "Truth" (The Reward Matrix)
# This calculates: (Price at Decision+1+5 / Price at Decision+1) - 1
new_master_engine.precompute_reward_matrix(holding_period=holding_pd)

print(f"✅ Reward Matrix Generated. Shape: {new_master_engine.reward_matrix.shape}")

# 3. NOW Initialize the Environment
# We pass the newly baked matrix into the stateful Arena
env = DiscoveryEnv(
    feature_cube=final_cache_df,
    reward_matrix=new_master_engine.reward_matrix,
    calendar=new_master_engine.trading_calendar,
    holding_period=holding_pd,
)

print("🚀 DiscoveryEnv Ready for Audit.")

In [ ]:
print(list(engine_out.perf_metrics.keys()))

In [ ]:
# --- THE FINAL PARITY AUDIT ---
test_date = pd.Timestamp("2025-01-02")
target_metric = "21d_Sharpe_ATRP"  # Verified Slugified Name

# 1. Build the One-Hot Action
# Index: 0 to 32 are metrics. 33 and 34 are Rank params.
metric_idx = final_cache_df.columns.get_loc(target_metric)
action = np.zeros(final_cache_df.shape[1] + 2)
action[metric_idx] = 1.0
action[-2], action[-1] = -1.0, 1.0  # Top 10 Tickers

# 2. Execute Step in the Arena
env.reset(start_date=test_date)
_, env_log_reward, _, info = env.step(action)

# 3. Execute the 'Old Guard' Human Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker="SPY",
    start_date=test_date,
    lookback_period=21,
    holding_period=holding_pd,
    metric="Sharpe (ATRP)",  # The original registry name
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# 4. Compare arithmetic vs log
eng_log_reward = engine_out.perf_metrics["holding_p_gain"]


print(f"\n--- AUDIT RESULT for {test_date.date()} ---")
print(f"Env Log Reward:     {env_log_reward:.12f}")
print(f"Engine Log Reward:  {eng_log_reward:.12f}")
print(f"Difference:         {abs(env_log_reward - eng_log_reward):.12f}")

# 5. Save the Excel Verification File
# This joins the Engine's Raw Scores with our Cache's Z-Scores
audit_df = engine_out.debug_data["selection_audit"].join(
    final_cache_df.xs(test_date, level="Date")[[target_metric]], how="inner"
)
audit_df.to_csv("Final_Absolute_Zero_Audit.csv")
print(f"💾 Final Audit Saved: 'Final_Absolute_Zero_Audit.csv'")

In [ ]:
from pathlib import Path

# 1. Define your pathing structure
# This ensures it works whether you are on a Local PC or Google Colab
cache_dir = Path("cache")
cache_file = cache_dir / "AlphaCache_2025_V1.pkl"

# 2. Safety First: Create the directory if it doesn't exist
# parents=True means it will create the whole path (e.g., data/discovery/cache)
# exist_ok=True means it won't crash if the folder is already there
cache_dir.mkdir(parents=True, exist_ok=True)

# 3. Save the 'Feature Cube'
# We use pickle because it preserves the Pandas MultiIndex and dtypes perfectly
try:
    cache.feature_cube.to_pickle(cache_file)
    print(f"✅ Persistence Successful!")
    print(f"💾 Path: {cache_file.absolute()}")
    print(f"📊 Cube Size: {cache.feature_cube.memory_usage().sum() / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Persistence Failed: {e}")

# 4. Quick Verify (The Trust Check)
# Ensure we can read it back immediately
import pandas as pd

test_load = pd.read_pickle(cache_file)
assert test_load.shape == cache.feature_cube.shape
print("🧪 Load-Back Test: PASSED. The data is safe.")

In [ ]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(new_master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

In [ ]:
# 1. Setup Parameters
test_date = pd.Timestamp("2025-01-02")
target_metric_full = "21d_Sharpe (ATRP)"  # The Cache Name
target_metric_base = "Sharpe (ATRP)"  # The Engine Name
benchmark_ticker = "SPY"


print(f"🕵️ Starting Deep Audit for {test_date.date()}...")

# --- STEP A: THE HUMAN ENGINE (RAW TRUTH) ---
# We run the engine in 'debug' mode to get the full universe ranking
engine_input = EngineInput(
    benchmark_ticker="SPY",
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric=target_metric_base,
    rank_start=1,
    rank_end=10,
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# Get the full universe scores before slicing (from debug data)
raw_scores_df = engine_out.debug_data["selection_audit"]
# Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
    columns={"Strategy_Score": "Engine_Raw_Value"}
)

# --- STEP B: THE AI CACHE (Z-SCORED VISION) ---
# Grab the specific date slice from the cache
cache_slice = cache.get_vision(test_date)
# Get the specific Z-scored column
cache_audit = cache_slice[[target_metric_full]].rename(
    columns={target_metric_full: "Cache_Z_Score"}
)

# --- STEP C: THE MERGE (THE EXCEL MOMENT) ---
# Combine both views on the Ticker index
verification_df = engine_audit.join(cache_audit, how="inner")

# Add a 'Rank' column based on the Raw Value to verify sorting
verification_df["Engine_Rank"] = verification_df["Engine_Raw_Value"].rank(
    ascending=False
)

# Add a 'Rank' column based on the Z-Score (They should be IDENTICAL)
verification_df["Cache_Rank"] = verification_df["Cache_Z_Score"].rank(ascending=False)

# --- STEP D: THE REWARD VERIFICATION ---
# Get the reward calculated by the environment
one_hot_action = np.zeros(cache.feature_cube.shape[1] + 2)
one_hot_action[cache.feature_cube.columns.get_loc(target_metric_full)] = 1.0
one_hot_action[-2], one_hot_action[-1] = -1.0, 1.0  # Offset 0, Width 10

env.reset(start_date=test_date)
_, env_reward, _, env_info = env.step(one_hot_action)

# # --- FINAL OUTPUT ---
# print(f"✅ Audit Complete. Verification File Generated.")
# print(f"Engine Reward: {engine_out.perf_metrics['p_hold_ret_ann']:.6f} (Annualized)")
# print(f"Env Log Reward: {env_reward:.6f} (5-day Log)")

# --- FINAL OUTPUT (FIXED) ---
print("\n--- PERFORMANCE AUDIT ---")
# 1. Inspect what the Engine actually calculated
print(f"Available Metric Keys: {list(engine_out.perf_metrics.keys())}")

# 2. Get the Arithmetic Return from the Engine
# Based on our AlphaEngine logic, the key is likely 'p_hold_ret'
engine_arith_ret = engine_out.perf_metrics.get("holding_p_gain", 0.0)

# 3. Calculate what the Log Reward SHOULD be based on that Engine return
expected_log_reward = np.log1p(engine_arith_ret)

print(f"Engine Raw Period Return (Arithmetic): {engine_arith_ret:.6f}")
print(f"Expected Log Reward [LN(1+R)]:      {expected_log_reward:.6f}")
print(f"Environment Log Reward (Actual):     {env_reward:.6f}")

# 4. Precision Check
diff = abs(expected_log_reward - env_reward)
if diff < 1e-10:
    print(
        "✅ REWARD PARITY: The environment reward is a perfect Log-Mirror of the engine."
    )
else:
    print(f"⚠️ REWARD DISCREPANCY: Difference of {diff:.12f}")


# Save to CSV
verification_df.to_csv("AbsoluteZero_Verification_Audit.csv")
print(f"💾 File saved as 'AbsoluteZero_Verification_Audit.csv'. Open this in Excel!")

In [ ]:
list(cache_slice.columns)

In [ ]:
# # Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
# engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
#     columns={"Strategy_Score": "Engine_Raw_Value"}
# )
print(f"raw_scores_df:\n{raw_scores_df.head()}\n")
print(f"engine_audit:\n{engine_audit.head()}\n")
print(f"cache_slice:\n{cache_slice}\n")
print(f"cache_slice.colums:\n{cache_slice.colums}\n")

In [ ]:
list(engine_out.perf_metrics.keys())

In [ ]:
# engine_out.perf_metrics["holding_p_gain"]
_x = engine_out.perf_metrics.get("holding_p_gain")
print(_x)

In [ ]:
# 1. Setup Parameters
decision_dt = pd.Timestamp("2025-12-10")
holding_pd = 5
lookback = 21
lookback_list = [21, 63, 189]
ticker = "NVDA"  # Let's test with the ticker you verified

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = new_master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = new_master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

In [ ]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = new_master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = new_master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

In [ ]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = new_master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

In [ ]:
# # --- TEST 3 (REVISITED): VERBOSE DISCOVERY VERIFICATION ---
# # decision_dt = pd.Timestamp("2025-12-10")
# # lookback = 21

# # 1. Setup 'One-Hot' Action for Sharpe (ATRP)
# registry_keys = list(METRIC_REGISTRY.keys())
# action_weights = np.zeros(len(registry_keys))
# action_weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# # 2. Run Discovery
# discovery = new_master_engine.run_discovery_action(
#     decision_dt, lookback, holding_period=5, weights=action_weights
# )

# # 3. Get UI Result for verification
# ui_result = new_master_engine.run(
#     EngineInput(
#         mode="Ranking",
#         start_date=decision_dt,
#         lookback_period=lookback,
#         holding_period=5,
#         metric="Sharpe (ATRP)",
#         benchmark_ticker="SPY",
#     )
# )

# print(f"=== DISCOVERY ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
# print(f"Target Strategy Weights: {discovery.action_weights}")
# print(f"Selected Tickers: {discovery.selected_tickers}")
# print("-" * 50)
# print(f"Internal Discovery Score (Top 1): {discovery.metric_values.iloc[0]:.4f}")
# print(
#     f"Original UI Strategy Value (Top 1): {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
# )
# print("-" * 50)
# print(f"VERITABLE REWARD (Sharpe): {discovery.veritable_reward:.4f}")
# print(f"UI REWARD (Sharpe):        {ui_result.perf_metrics['holding_p_sharpe']:.4f}")

# if discovery.selected_tickers == ui_result.tickers:
#     print("\n✅ SELECTION MATCH: The agent and UI chose the same tickers.")

In [ ]:
# --- STEP 4.1: THE VERITABLE STANDARD PROOF ---

# # 1. Setup Parameters
# decision_dt = pd.Timestamp("2025-12-10")
# holding_pd = 5
# lookback = 21
# ticker = "SHV"  # Let's test with the ticker you verified

# 2. METHOD A: The Verified UI Engine (Compounded Daily Returns)
# This is the code you already verified with Excel
ui_result = new_master_engine.run(
    EngineInput(
        mode="Manual List",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=holding_pd,
        metric="Price Gain",
        manual_tickers=[ticker],
        benchmark_ticker="SPY",
    )
)
buy_date = ui_result.buy_date
end_date = ui_result.holding_end_date
ui_gain = ui_result.perf_metrics["holding_p_gain"]

# 3. METHOD B: The New Vectorized Shifter (Price Ratio)
# We calculate the log-gain directly from the price matrix
# log(P_end / P_start)
price_start = new_master_engine.df_close.loc[buy_date, ticker]
price_end = new_master_engine.df_close.loc[end_date, ticker]
vectorized_gain = np.log(price_end / price_start)

# 4. FINAL COMPARISON
print(f"=== VERITABLE STANDARD PROOF ({ticker}) ===")
print(f"Buy Date: {buy_date.date()} | End Date: {end_date.date()}")
print(f"Price at Buy: {price_start:.4f}")
print(f"Price at End: {price_end:.4f}")
print("-" * 40)
print(f"Method A (UI Engine Gain):   {ui_gain:.8f}")
print(f"Method B (Shifted Matrix):   {vectorized_gain:.8f}")
print("-" * 40)

diff = abs(ui_gain - vectorized_gain)
if diff < 1e-10:
    print("✅ VERIFICATION SUCCESS: Both methods are mathematically identical.")
    print("The Vectorized 'Time Machine' is now safe to use for RL Training.")
else:
    print(f"❌ VERIFICATION FAILED: Difference of {diff:.10f} detected.")

#

In [ ]:
# --- STEP 4.2: VECTORIZED DISCOVERY VERIFICATION ---

# A. Initialize the Time Machine (Do this once)
new_master_engine.precompute_reward_matrix(holding_period=5)

# decision_dt = pd.Timestamp("2025-12-10")
# lookback = 21

# B. Setup Action (Sharpe ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
weights = np.zeros(len(registry_keys))
weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# C. Run Discovery (Using the NEW Vectorized functions)
# CHANGE: Only one variable on the left side
discovery = new_master_engine.run_discovery_action(
    decision_dt, lookback, holding_pd, weights=weights
)

# Access the matrix via the object property
raw_matrix = discovery.raw_alpha_matrix

print(f"SHV Raw Sharpe: {raw_matrix.loc['SHV', 'Sharpe (ATRP)']:.4f}")


# D. Get UI Result for the "Gold Standard"
ui_result = new_master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== VECTORIZED ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(
    f"Selected Tickers: {discovery.selected_tickers[:3]}... (Match UI: {discovery.selected_tickers == ui_result.tickers})"
)
print("-" * 60)

# This confirms the 0.8381 'Signal' is preserved
print(f"SIGNAL CHECK (Lookback):")
print(f"  Discovery Score (Top 1):    {discovery.metric_values.iloc[0]:.4f}")
print(
    f"  Original UI Strategy Val:   {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)

print("-" * 60)

# This confirms the 'Reward' matches the price action
# Note: UI Reward is Sharpe (28.29), Discovery Reward is now Total Return for RL efficiency
ui_return = ui_result.perf_metrics["holding_p_gain"]

print(f"REWARD CHECK (Holding Period Return):")
print(f"  Vectorized Reward:          {discovery.veritable_reward:.8f}")
print(f"  UI Verified Gain:           {ui_return:.8f}")

if abs(discovery.veritable_reward - ui_return) < 1e-7:
    print("\n✅ VERITABLE MATCH: The Time Machine matches the UI reality.")
    print("The Engine is now ready for High-Frequency Training.")

In [ ]:
# --- ENSEMBLE VERIFICATION TEST ---
# decision_dt = pd.Timestamp("2025-12-10")
# lookback_list = [21, 63, 189]

# 1. Generate the Ensemble
ensemble_vision = new_master_engine.compute_alpha_ensemble(decision_dt, lookback_list)

print(f"=== ENSEMBLE VISION SUMMARY (Date: {decision_dt.date()}) ===")
print(f"Total Features per Ticker: {ensemble_vision.shape[1]}")
print(f"Resolutions: {lookback_list}")
print("-" * 50)

# 2. Verify Component Integrity
# Let's check NVDA's Sharpe (ATRP) across the resolutions
nvda_stats = ensemble_vision.loc["NVDA"]

# Grab the 21d result to check against our 0.8410 benchmark
# Note: This will be the Z-scored/Clipped value, not raw 0.8410
val_21d = nvda_stats.get("21d_Sharpe (ATRP)")
val_189d = nvda_stats.get("189d_Sharpe (ATRP)")

print(f"NVDA 21d Sharpe (Z-Score):  {val_21d:.4f}")
print(f"NVDA 189d Sharpe (Z-Score): {val_189d:.4f}")

# 3. Check for Data Integrity
# Ensure we don't have all-NaN columns or duplicate data
if ensemble_vision.isna().sum().sum() == 0:
    print("\n✅ DATA INTEGRITY: Ensemble is fully populated (No NaNs).")

if val_21d != val_189d:
    print("✅ TEMPORAL DIFFERENTIATION: 21d and 189d signals are distinct.")
else:
    print("❌ ERROR: Temporal signals are identical (Slicing error?)")

# Display a sample of the 'Alpha Tensor' for one ticker
print("\nSample Feature Vector (NVDA):")
print(nvda_stats.head(10))

In [ ]:
# Check if the DYNAMIC metrics differ across resolutions
print(f"NVDA 21d Price Gain (Z):  {ensemble_vision.loc['NVDA', '21d_Price Gain']:.4f}")
print(f"NVDA 189d Price Gain (Z): {ensemble_vision.loc['NVDA', '189d_Price Gain']:.4f}")

if (
    ensemble_vision.loc["NVDA", "21d_Price Gain"]
    != ensemble_vision.loc["NVDA", "189d_Price Gain"]
):
    print(
        "\n✅ SLICING VERIFIED: Dynamic window metrics are correctly calculating different time horizons."
    )

In [ ]:
raw_matrix.loc["NVDA"]

In [ ]:
one_year_ago = decision_dt - pd.DateOffset(years=1)
one_year_ago

In [ ]:
# 1. Initialize Cache for a specific window
cache = AlphaCache(new_master_engine, lookbacks=lookback_list)

# We start in 2024 so the Agent has a year of "warm up" before 2025
one_year_ago = decision_dt - pd.DateOffset(years=1)
cache.build(start_date=one_year_ago)

# 2. Initialize the Optimized Environment
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=holding_pd)

# 3. Benchmark Step Speed
import time

obs = env.reset(start_date=decision_dt)
action_size = (3 * 11) + 2
random_action = np.random.uniform(-1, 1, size=action_size)

print("\n⏱️ Starting High-Frequency Benchmark...")
start_time = time.time()
n_steps = 100

for i in range(n_steps):
    obs, reward, done, info = env.step(random_action)
    if done:
        env.reset()

end_time = time.time()
total_time = end_time - start_time

avg_step = total_time / n_steps

print(f"✅ Finished {n_steps} steps in {total_time:.4f} seconds.")
print(f"🚀 Average Step Time: {avg_step:.6f} seconds.")
print(
    f"📈 Projected Time for 1,000,000 steps: { (avg_step * 1_000_000) / 3600:.2f} hours (Previously: 9,700 hours)"
)

In [ ]:
from pathlib import Path

# 1. Define your pathing structure
# This ensures it works whether you are on a Local PC or Google Colab
cache_dir = Path("cache")
cache_file = cache_dir / "AlphaCache_2025_V1.pkl"

# 2. Safety First: Create the directory if it doesn't exist
# parents=True means it will create the whole path (e.g., data/discovery/cache)
# exist_ok=True means it won't crash if the folder is already there
cache_dir.mkdir(parents=True, exist_ok=True)

# 3. Save the 'Feature Cube'
# We use pickle because it preserves the Pandas MultiIndex and dtypes perfectly
try:
    cache.feature_cube.to_pickle(cache_file)
    print(f"✅ Persistence Successful!")
    print(f"💾 Path: {cache_file.absolute()}")
    print(f"📊 Cube Size: {cache.feature_cube.memory_usage().sum() / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Persistence Failed: {e}")

# 4. Quick Verify (The Trust Check)
# Ensure we can read it back immediately
import pandas as pd

test_load = pd.read_pickle(cache_file)
assert test_load.shape == cache.feature_cube.shape
print("🧪 Load-Back Test: PASSED. The data is safe.")

In [ ]:
# Diagnostic: List all 33 feature names
print("Current AlphaCache Feature Names:")
for i, col in enumerate(cache.feature_cube.columns):
    print(f"{i:02d}: {col}")

In [ ]:
# # This logic doesn't care if there's a space or an underscore!
# target_search = ["21d", "Sharpe", "ATRP"]
# matching_cols = [
#     c for c in cache.feature_cube.columns if all(word in c for word in target_search)
# ]

In [ ]:
# Re-initialize the env with the new class definition
env = DiscoveryEnv(new_master_engine, cache, holding_period=5)

# --- ONE-HOT PARITY TEST ---
# Search for '21d' and 'Sharpe' and '(ATRP)'
target_search = ["21d", "Sharpe", "(ATRP)"]
matching_cols = [
    c for c in cache.feature_cube.columns if all(word in c for word in target_search)
]

if not matching_cols:
    raise ValueError(
        f"❌ Search failed. Cols are: {cache.feature_cube.columns.tolist()[:5]}..."
    )

target_metric = matching_cols[0]
metric_idx = cache.feature_cube.columns.get_loc(target_metric)

# Build the One-Hot Action
action_size = cache.feature_cube.shape[1] + 2
one_hot_action = np.zeros(action_size)
one_hot_action[metric_idx] = 1.0  # Turn on ONLY this metric
one_hot_action[-2] = -1.0  # Offset = 0
one_hot_action[-1] = 1.0  # Width = 10 (Max)

# Execute Step
test_date = pd.Timestamp("2025-01-02")
env.reset(start_date=test_date)
_, _, _, info = env.step(one_hot_action)
env_tickers = info["tickers"]

# Run AlphaEngine Comparison
# Note: we pass the actual column name to see if the registry recognizes it
old_input = EngineInput(
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",  # Standardize based on your column names
    rank_start=1,
    rank_end=10,
    benchmark_ticker="SPY",
)
old_output = new_master_engine.run(old_input)
engine_tickers = old_output.tickers

print(f"\n--- PARITY TEST: {target_metric} ---")
print(f"Environment: {env_tickers}")
print(f"AlphaEngine: {engine_tickers}")

if set(env_tickers) == set(engine_tickers):
    print("✅ PARITY ACHIEVED!")
else:
    print(
        "❌ Content mismatch. Check if AlphaEngine 'metric' name matches the feature."
    )

In [ ]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(new_master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

In [ ]:
# --- STEP 5: OPTIMIZED DISCOVERY WALK ---
from core.environment import DiscoveryEnv

# 1. Initialize Gym using the CACHE (The high-speed version)
# Note: We don't pass 'lookbacks' here anymore because they are inside the cache!
env = DiscoveryEnv(
    engine=new_master_engine,
    cache=cache,
    holding_period=5,  # <--- This is the key change
)

# 2. Define the 'Discovery Loop'
obs = env.reset(start_date=pd.Timestamp("2025-01-02"))
done = False
total_steps = 0

print(f"🚀 Starting High-Speed Discovery Walk from {obs['date'].date()}...")

# Action Size: (Lookbacks * 11 Metrics) + 2 Rank Params
# Our cache has 33 features (3 lookbacks * 11 metrics)
action_size = cache.feature_cube.shape[1] + 2

while not done:
    # Agent outputting random weights
    random_action = np.random.uniform(-1, 1, size=action_size)

    # This call is now 10,000x faster!
    obs, reward, done, info = env.step(random_action)

    if total_steps % 10 == 0:
        print(
            f"Step {total_steps:03d} | Date: {info['date'].date()} | Reward: {reward:+.4f} | Tickers: {len(info.get('tickers', []))}"
        )

    total_steps += 1

print("-" * 50)
final_return = (env.equity_curve[-1] - 1) * 100
print(f"✅ Discovery Walk Complete. Total Steps: {total_steps}")
print(f"Final Agent Equity: {env.equity_curve[-1]:.4f} ({final_return:+.2f}%)")

In [ ]:
print(f"action_size:\n{action_size}\n")
print(f"random_action :\n{random_action }\n")
print(f"obs:\n{obs}\n")
print(f"reward:\n{reward}\n")
print(f"done:\n{done}\n")
print(f"info:\n{info}\n")
print(f"env.equity_curve:\n{env.equity_curve}\n")

In [ ]:
# 1. Pick a random sample from the cache
sample_idx = 100
sample_date = cache.feature_cube.index.get_level_values("Date")[sample_idx]
sample_ticker = cache.feature_cube.index.get_level_values("Ticker")[sample_idx]

# 2. Get the value from the Cache
cached_val = cache.feature_cube.loc[(sample_date, sample_ticker)].iloc[0]

# 3. Get the value from the Engine (The slow way)
# We calculate just for this one date
engine_ensemble = new_master_engine.compute_alpha_ensemble(sample_date, [21, 63, 252])
engine_val = engine_ensemble.loc[sample_ticker].iloc[0]

print(f"Verification for {sample_ticker} on {sample_date.date()}:")
print(f"  Cache Value:  {cached_val:.6f}")
print(f"  Engine Value: {engine_val:.6f}")
print(f"  Difference:   {abs(cached_val - engine_val):.12f}")

assert np.isclose(cached_val, engine_val, atol=1e-8), "❌ Cache Integrity Failed!"
print("✅ Cache Integrity Verified. The fast data is identical to the trusted data.")

In [ ]:
import time

# 1. Setup
lookbacks = [21, 63, 252]
cache = AlphaCache(new_master_engine, lookbacks)
cache.build()  # This takes a minute, but only runs ONCE

env = DiscoveryEnv(new_master_engine, cache)

# 2. Benchmark the Step
obs = env.reset()
action = np.random.uniform(-1, 1, size=(len(lookbacks) * 11 + 2))

start_time = time.time()
for _ in range(100):
    _, _, done, _ = env.step(action)
    if done:
        env.reset()
end_time = time.time()

avg_step_time = (end_time - start_time) / 100
print(f"⏱️ New Average Step Time: {avg_step_time:.6f} seconds")
print(f"🚀 Speedup Factor: {35 / avg_step_time:.0f}x")

In [ ]:
obs

In [ ]:
discovery

In [ ]:
# ui_result.perf_metrics
# registry_keys
# sharpe_atrp_idx
action_weights

In [ ]:
macro_df.loc[decision_dt]

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    new_master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    new_master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [ ]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])